In [1]:
!pip install datasets

In [1]:
from datasets import load_dataset
dataset = load_dataset("knkarthick/dialogsum")

README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [3]:
dataset['train'][1]['dialogue']

"#Person1#: Hello Mrs. Parker, how have you been?\n#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.\n#Person1#: Very well. Let's see, according to his vaccination record, Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for Hepatitis A, Chickenpox and Measles shots.\n#Person2#: What about Rubella and Mumps?\n#Person1#: Well, I can only give him these for now, and after a couple of weeks I can administer the rest.\n#Person2#: OK, great. Doctor, I think I also may need a Tetanus booster. Last time I got it was maybe fifteen years ago!\n#Person1#: We will check our records and I'll have the nurse administer and the booster as well. Now, please hold Ricky's arm tight, this may sting a little."

In [4]:
dataset['train'][1]['summary']

'Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.'

In [5]:
!pip install transformers

#### Before Fine tuning

In [6]:
from transformers import BartTokenizer, BartForConditionalGeneration

tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [7]:
article_1=dataset['train'][1]['dialogue']

In [8]:
inputs = tokenizer(article_1, return_tensors="pt", max_length=1024, truncation=True)

# Generate summary IDs
summary_ids = model.generate(inputs["input_ids"], num_beams=4, max_length=100, min_length=30)

# Decode IDs back to readable text
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(summary)

Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for Hep atitis A, Chickenpox and Measles shots.


#### With Finetuning

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

In [10]:
def preprocess_function(batch):
  source=batch['dialogue']
  target=batch['summary']
  source_ids=tokenizer(source,padding='max_length',truncation=True,max_length=128)
  target_ids=tokenizer(target,padding='max_length',truncation=True,max_length=128)

  labels=target_ids['input_ids']
  labels=[[(label if label!=tokenizer.pad_token_id else -100) for label in labels_example]for labels_example in labels]

  return{
      "input_ids":source_ids['input_ids'],
      "attention_mask":source_ids['attention_mask'],
      "labels":labels
  }

In [11]:
df_source=dataset.map(preprocess_function,batched=True)

Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [12]:
from transformers import TrainingArguments,Trainer

training_args=TrainingArguments(
    output_dir='.results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    remove_unused_columns=True
)

In [13]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=df_source['train'],
    eval_dataset=df_source['test']
)

In [ ]:
trainer.train()

Step,Training Loss
500,1.591923
1000,1.488137
1500,1.433559


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
eval_results=trainer.evaluate()

In [ ]:
eval_results

#### Saving the model

In [ ]:
model.save_pretraoned('bart_model')
tokenizer.save_pretrained('bart_tokenizer')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

def summarize(blog_post):
  inputs=tokenizer(blog_post,return_tensors='pt',max_length=1024,truncation=True)
  summary_ids=model.generate(inputs['input_ids'],num_beams=4,max_length=100,min_length=30)
  summary=tokenizer.decode(summary_ids[0],skip_special_tokens=True)
  return summary

In [ ]:
blog_post=" "
summary=summarize(blob_post)
print(f"Summary: {summary}")